# 04 — Exploration et comparaison de détecteurs d'anomalies

**Projet** : Détection de fraude sur transactions de paiement avec scikit-learn
**Modèle configuré** : `isolation_forest` (Forêt d'isolation (Isolation Forest))
**Pourquoi ce choix** : La forêt d'isolation est le détecteur de référence pour un premier système de scoring de fraude : elle ne fait **aucune hypothèse de distribution**, produit un score continu et ordonnable (indispensable pour piloter au budget), passe à l'échelle (entraînement linéaire, inférence en quelques millisecondes par transaction) et reste déterministe à graine fixée. Son principe — une anomalie est plus facile à isoler par des coupes aléatoires — correspond exactement à la structure de la fraude bancaire : vélocité explosive, empreinte d'appareil neuve, géographie incohérente sont des points peu denses. Ce projet documente aussi ses limites : elle est aveugle à la **fraude amicale** (transaction légitime dans sa forme) et sensible aux variables non pertinentes quand `max_features` est trop élevé. La comparaison avec un one-class SVM et un Elliptic Envelope, au notebook 04, montre ce que l'on gagne (robustesse, coût) et ce que l'on perd (frontière de densité explicite).

En détection d'anomalies non supervisée, la question « pourquoi cet algorithme ? » se dédouble :

* **quel score ?** — tous les détecteurs ne rendent pas la même quantité : une forêt d'isolation
  produit un score d'anormalité continu, un auto-encodeur une erreur de reconstruction, un
  one-class SVM une distance à la frontière. Seul un **score ordonnable** permet de piloter au
  budget ;
* **quel point de fonctionnement ?** — la « contamination » déclarée fixe un seuil par défaut, mais
  la vraie contrainte est la **capacité d'investigation** des analystes.

Ce notebook répond aux deux **par des chiffres** : plancher aléatoire, règles métier existantes,
comparaison d'algorithmes, sensibilité à la contamination, stabilité du classement entre graines.

## Objectifs pédagogiques

1. Commencer par le **plancher** : un score aléatoire obtient une PR AUC égale à la prévalence, et les règles métier existantes donnent la référence à battre.
1. Comparer les algorithmes de la stack **à données et budget égaux** : la comparaison n'a de sens qu'à protocole identique.
1. Comprendre que la **contamination** déplace le seuil sans changer le classement — donc pourquoi on pilote au budget.
1. Mesurer la **stabilité du classement** entre graines : une file d'alertes qui change tous les jours n'est pas exploitable.

**Objectifs transverses du dépôt**

- Comprendre pourquoi la fraude se détecte sans supervision : étiquette tardive (chargeback à J+30), partielle et biaisée par les règles existantes.
- Construire un pipeline sans fuite : `is_fraud` et `fraud_scheme` sont des métadonnées exclues des features par configuration.
- Lire les bonnes métriques en forte imbalance : PR AUC et lift plutôt qu'accuracy et ROC AUC, et toujours relativement au plancher (= prévalence).

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet configure loguru au premier `get_logger()` appelé par `src`. On prend la main ici,
# au niveau WARNING : sans cela, chaque cellule d'entraînement noierait ses tableaux sous les
# lignes INFO de production. Les avertissements réels restent visibles — c'est l'essentiel.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (12000 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 12000

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.42)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": list(pipeline.feature_names_out),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

## 0. Le détecteur configuré

On entraîne d'abord le modèle déclaré dans `conf/model/default.yaml` : il servira de référence
dans toutes les sections suivantes (comparaison aux règles métier, stabilité, sensibilité).

In [ ]:
from src.models import build_model

# Tâche non supervisée : `fit` reçoit `None` en guise de cible. Le modèle apprend la région de
# densité « normale » sur le train uniquement ; la validation sert à choisir le point de
# fonctionnement, jamais à ajuster les hyperparamètres sur les fraudes.
MODEL = build_model(CONFIG, feature_names=PREPARED["feature_names"])
# `build_model(params=...)` **remplace** les réglages de `conf/model/default.yaml`. Pour ne faire
# varier qu'un seul facteur à la fois dans les sections suivantes (graine, contamination, grille),
# on part donc toujours de cette copie des réglages configurés.
CONFIGURED_PARAMS = dict(CONFIG.model.params)
FIT_RESULT = MODEL.fit(PREPARED["X_train"], None, X_val=PREPARED["X_val"], y_val=None, callbacks=[])
print(MODEL.summary())
print(f"entraînement : {FIT_RESULT.duration_seconds:.2f} s")

## 1. Le plancher d'abord : score aléatoire et règles métier

In [ ]:
from src.training.losses_metrics import MetricCalculator, MetricInputs

RANKING_METRICS = [
    name
    for name in [CONFIG.metrics.primary, *CONFIG.metrics.secondary]
    if name in {"pr_auc", "roc_auc", "recall_at_budget", "precision_at_budget"}
]

val_frame = PREPARED["enriched"]["val"]
if val_frame is None or PREPARED["X_val"] is None:
    raise RuntimeError(
        "Split de validation absent : renseignez `train.split.val_size` dans conf/config.yaml"
    )
VAL_LABELS = val_frame["is_fraud"].to_numpy().astype(int)
# Le point de fonctionnement est **lu dans la configuration** (noeud `fraud_detection` de
# `conf/config.yaml`) : le notebook et le pipeline d'inférence pilotent au même budget.
BUDGET_RATE = float(dict(getattr(CONFIG, "fraud_detection", {}) or {}).get("budget_rate", 0.02))
VAL_BUDGET = max(1, round(BUDGET_RATE * len(VAL_LABELS)))
calculator = MetricCalculator(
    task=CONFIG.metrics.task, metrics=RANKING_METRICS, extra={"budget": VAL_BUDGET}
)

prevalence = float(VAL_LABELS.mean())
print(f"transactions de validation : {len(VAL_LABELS)}")
print(f"fraudes confirmées         : {int(VAL_LABELS.sum())}")
print(f"prévalence                 : {prevalence:.3%}")
print(f"budget d'investigation     : {VAL_BUDGET} alertes ({BUDGET_RATE:.1%} du flux)")
print(f"plancher PR AUC            : {prevalence:.4f}")

rng = np.random.default_rng(0)
amount = pd.to_numeric(val_frame["amount_eur"], errors="coerce").fillna(0.0).to_numpy()
velocity = pd.to_numeric(val_frame["transactions_24h"], errors="coerce").fillna(0.0).to_numpy()
failures = pd.to_numeric(val_frame["failed_attempts_1h"], errors="coerce").fillna(0.0).to_numpy()

references = {
    "score aléatoire (plancher)": rng.random(len(VAL_LABELS)),
    "règle métier : montant > P99": (amount > np.quantile(amount, 0.99)).astype(float),
    "règle métier : vélocité 24 h >= 5": (velocity >= 5).astype(float),
    "règle métier : échecs 1 h >= 2": (failures >= 2).astype(float),
    "règles combinées (OU)": (
        (amount > np.quantile(amount, 0.99)) | (velocity >= 5) | (failures >= 2)
    ).astype(float),
    # Le détecteur configuré rejoint le tableau : c'est la comparaison qui décide. Un modèle qui
    # ne bat pas les règles métier qu'il est censé remplacer ne doit pas être mis en production.
    f"détecteur configuré ({MODEL.algorithm})": np.asarray(
        MODEL.predict(PREPARED["X_val"]), dtype="float64"
    ).ravel(),
}

rows = []
for name, scores in references.items():
    values = calculator.evaluate(MetricInputs(y_true=VAL_LABELS, y_pred=None, y_proba=scores))
    rows.append(
        {
            "référence": name,
            **{key: round(float(value), 4) for key, value in values.items()},
        }
    )
baseline_table = pd.DataFrame(rows)
display(baseline_table)

**Ce qu'il faut retenir**

- Le score aléatoire donne une PR AUC égale à la prévalence : c'est le **plancher**. Toute valeur publiée doit être comparée à lui, jamais lue dans l'absolu.
- Les règles métier (montant > P99, vélocité ≥ 5, échecs ≥ 2) ont une précision correcte mais un rappel faible : elles ne capturent que les schémas déjà vus. C'est exactement la limite que le détecteur appris doit repousser.
- La règle « montant > P99 » illustre le piège des outliers légitimes : elle alerte sur les achats de luxe, qui ne sont pas de la fraude.
- La métrique de décision du projet est `pr_auc` (sens : maximize) ; le seuil de qualité déclaré est 0.42.

## 2. Comparaison des algorithmes de la stack

In [ ]:
import warnings

from src.models.factory import available_algorithms

ALGORITHMS = available_algorithms(CONFIG.metrics.task)
print(f"{len(ALGORITHMS)} algorithmes disponibles pour la tâche '{CONFIG.metrics.task}' :")
print(ALGORITHMS)

rows = []
warnings_par_algorithme: dict[str, dict[str, int]] = {}
# Comparaison **loyale** : `params={}` construit chaque algorithme avec ses réglages par défaut.
# Les `model.params` configurés sont propres à la forêt d'isolation (`n_estimators`,
# `max_samples`, `max_features`) : les injecter dans un one-class SVM lèverait une erreur, et les
# régler « à la main » pour chaque concurrent fausserait le classement. Le détecteur configuré et
# réglé est, lui, évalué en section 1.
for algorithm in ALGORITHMS:
    try:
        # Les avertissements des bibliothèques sont **capturés puis restitués** en fin de cellule :
        # les masquer cacherait une information diagnostique (covariance dégénérée, convergence),
        # les laisser inonder la sortie noierait le tableau. Ni l'un ni l'autre.
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter("always")
            candidate = build_model(
                CONFIG, feature_names=PREPARED["feature_names"], algorithm=algorithm, params={}
            )
            result = candidate.fit(
                PREPARED["X_train"], None, X_val=PREPARED["X_val"], y_val=None, callbacks=[]
            )
            scores = np.asarray(candidate.predict(PREPARED["X_val"]), dtype="float64").ravel()
        counts: dict[str, int] = {}
        for item in captured:
            first_line = str(item.message).strip().splitlines()[0][:78]
            key = f"{item.category.__name__} : {first_line}"
            counts[key] = counts.get(key, 0) + 1
        if counts:
            warnings_par_algorithme[algorithm] = counts
        values = calculator.evaluate(MetricInputs(y_true=VAL_LABELS, y_pred=None, y_proba=scores))
        row = {
            "algorithme": algorithm,
            CONFIG.metrics.primary: values.get(CONFIG.metrics.primary, float("nan")),
        }
        for name in RANKING_METRICS[1:]:
            row[name] = values.get(name, float("nan"))
        row["score moyen"] = round(float(np.mean(scores)), 4)
        row["secondes"] = round(result.duration_seconds, 2)
        rows.append(row)
    except Exception as error:  # un algorithme incompatible ne doit pas casser l'exploration
        rows.append(
            {
                "algorithme": algorithm,
                CONFIG.metrics.primary: float("nan"),
                "erreur": str(error)[:90],
            }
        )

ascending = CONFIG.metrics.direction == "minimize"
ranking = pd.DataFrame(rows).sort_values(
    CONFIG.metrics.primary, ascending=ascending, na_position="last"
)
display(ranking.round(4))

if warnings_par_algorithme:
    print("Avertissements émis pendant la comparaison (capturés, ni masqués ni perdus) :")
    for algorithm, counts in warnings_par_algorithme.items():
        for message, count in counts.items():
            print(f"  - {algorithm} ({count}x) {message}")

**Ce qu'il faut retenir**

- Les quatre détecteurs ne testent pas la même hypothèse : la forêt d'isolation cherche des points **faciles à isoler** par coupes aléatoires, le one-class SVM une **frontière de densité** à noyau RBF, l'enveloppe elliptique un **écart de Mahalanobis** à un centre robuste, et le LOF une rareté **relative au voisinage**.
- Chacun est évalué avec ses réglages par défaut (`params={}`) : c'est la comparaison loyale. Le détecteur configuré et réglé, lui, est mesuré en section 1 — un algorithme battu « par défaut » peut très bien gagner une fois réglé.
- Le LOF score bien sur un échantillon de quelques milliers de transactions, mais il doit **conserver le jeu d'entraînement** pour scorer une nouvelle ligne : coût mémoire et latence incompatibles avec un flux de millions de paiements. Le one-class SVM a le même défaut (coût quadratique).
- L'enveloppe elliptique déclenche un avertissement de covariance non plein rang : sur 51 colonnes dont une quarantaine de modalités one-hot quasi constantes, la covariance robuste est dégénérée. C'est une leçon générale — les détecteurs gaussiens supportent mal le one-hot, les arbres s'en accommodent.
- Un écart de PR AUC inférieur à 2x la dispersion entre graines (section 4) n'est pas un signal : ne pas choisir un algorithme sur un écart de cet ordre.

## 3. Sensibilité à la contamination déclarée

In [ ]:
# La « contamination » est la part d'anomalies que le détecteur *suppose* a priori. Elle déplace
# le seuil interne de décision, mais — point crucial — elle ne change pas le CLASSEMENT produit par
# le score continu. D'où l'intérêt de piloter au budget plutôt qu'au seuil par défaut.
CONTAMINATIONS = (0.005, 0.01, 0.02, 0.05, 0.10, 0.20)

rows = []
for contamination in CONTAMINATIONS:
    candidate = build_model(
        CONFIG,
        feature_names=PREPARED["feature_names"],
        params={**CONFIGURED_PARAMS, "contamination": contamination},
    )
    candidate.fit(PREPARED["X_train"], None, X_val=PREPARED["X_val"], y_val=None, callbacks=[])
    scores = np.asarray(candidate.predict(PREPARED["X_val"]), dtype="float64").ravel()
    values = calculator.evaluate(MetricInputs(y_true=VAL_LABELS, y_pred=None, y_proba=scores))
    flagged = (scores >= np.quantile(scores, 1 - contamination)).astype(int)
    rows.append(
        {
            "contamination": contamination,
            CONFIG.metrics.primary: values.get(CONFIG.metrics.primary, float("nan")),
            **{name: values.get(name, float("nan")) for name in RANKING_METRICS[1:]},
            "alertes au seuil interne": int(flagged.sum()),
            "précision au seuil interne": round(
                float((flagged & VAL_LABELS).sum() / max(int(flagged.sum()), 1)), 4
            ),
        }
    )

contamination_table = pd.DataFrame(rows)
display(contamination_table.round(4))

fig, axis = plt.subplots(figsize=(7.4, 4.0))
axis.plot(
    contamination_table["contamination"] * 100,
    contamination_table[CONFIG.metrics.primary],
    marker="o",
    color="#0a9396",
    label=CONFIG.metrics.primary,
)
axis.axhline(
    float(VAL_LABELS.mean()), color="#ae2012", ls="--", lw=1.2, label="plancher (prévalence)"
)
second = axis.twinx()
second.plot(
    contamination_table["contamination"] * 100,
    contamination_table["alertes au seuil interne"],
    color="#ee9b00",
    ls=":",
    marker="s",
    ms=4,
    label="alertes au seuil interne",
)
second.set_ylabel("nombre d'alertes", color="#ee9b00")
second.grid(False)
axis.set_xscale("log")
axis.set_xlabel("contamination déclarée (%)")
axis.set_ylabel(CONFIG.metrics.primary)
axis.set_title("La contamination déplace le seuil, pas le classement")
lines, labels_ = axis.get_legend_handles_labels()
extra_lines, extra_labels = second.get_legend_handles_labels()
axis.legend(lines + extra_lines, labels_ + extra_labels, fontsize=8, loc="center left")
fig.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- La PR AUC et le rappel **au budget** sont quasi constants : la contamination ne change pas le classement, seulement le seuil interne. C'est la justification du pilotage au budget.
- Le nombre d'alertes au seuil interne, lui, suit mécaniquement la contamination : déclarer 20 % d'anomalies dans un flux qui en contient 1,8 % sature l'équipe d'analyse.
- En production, la contamination se règle sur la **capacité d'investigation** (ici 2 % du flux), pas sur une estimation de la prévalence réelle — qui est inconnue par définition.

## 4. Stabilité : le classement survit-il à un changement de graine ?

In [ ]:
from scipy.stats import spearmanr

STABILITY_SEEDS = (7, 19, 123)
reference_scores = np.asarray(MODEL.predict(PREPARED["X_val"]), dtype="float64").ravel()
reference_top = set(np.argsort(-reference_scores, kind="stable")[:VAL_BUDGET].tolist())

rows = []
for seed in STABILITY_SEEDS:
    candidate = build_model(
        CONFIG,
        feature_names=PREPARED["feature_names"],
        params={**CONFIGURED_PARAMS, "random_state": seed},
    )
    candidate.fit(PREPARED["X_train"], None, X_val=PREPARED["X_val"], y_val=None, callbacks=[])
    scores = np.asarray(candidate.predict(PREPARED["X_val"]), dtype="float64").ravel()
    top = set(np.argsort(-scores, kind="stable")[:VAL_BUDGET].tolist())
    values = calculator.evaluate(MetricInputs(y_true=VAL_LABELS, y_pred=None, y_proba=scores))
    rows.append(
        {
            "graine": seed,
            CONFIG.metrics.primary: values.get(CONFIG.metrics.primary, float("nan")),
            "corrélation de rang (Spearman)": round(
                float(spearmanr(reference_scores, scores).statistic), 4
            ),
            "recouvrement du top-budget": round(len(top & reference_top) / max(len(top), 1), 4),
        }
    )

stability_table = pd.DataFrame(rows)
display(stability_table.round(4))
print(
    f"dispersion de la métrique primaire entre graines : "
    f"{float(stability_table[CONFIG.metrics.primary].std()):.4f}"
)

**Ce qu'il faut retenir**

- Le **recouvrement du top-budget** mesure ce que le métier redoute : une transaction alertée hier qui ne l'est plus aujourd'hui, à comportement inchangé, décrédibilise l'outil auprès des analystes.
- La corrélation de rang de Spearman est plus exigeante que le recouvrement du top-K : elle vérifie tout le classement, pas seulement la tête de file.
- Une forêt d'isolation est stabilisée en augmentant `n_estimators` ; un auto-encodeur en fixant toutes les graines (initialisation, découpage des lots) et en réduisant le taux d'apprentissage.

## 5. Sensibilité aux hyperparamètres

In [ ]:
import itertools

# Grille déclarée dans le manifeste du projet (`extras.notebook_param_grid`) et injectée ici au
# moment du build : le notebook reste une source unique de vérité côté configuration, et la grille
# explorée est versionnée avec le projet plutôt qu'improvisée à chaque exécution.
GRID = {"n_estimators": [200, 500], "max_samples": [0.5, 0.8], "max_features": [2, 5, 12]}
combinations = list(itertools.product(*[GRID[name] for name in GRID]))
print(f"{len(combinations)} combinaisons testées sur {list(GRID)}")

grid_rows = []
for combination in combinations:
    # Les paramètres balayés écrasent les réglages configurés ; les autres sont conservés à
    # l'identique, sinon chaque combinaison changerait plusieurs facteurs à la fois.
    params = {**CONFIGURED_PARAMS, **dict(zip(GRID, combination, strict=True))}
    candidate = build_model(CONFIG, feature_names=PREPARED["feature_names"], params=params)
    result = candidate.fit(
        PREPARED["X_train"], None, X_val=PREPARED["X_val"], y_val=None, callbacks=[]
    )
    scores = np.asarray(candidate.predict(PREPARED["X_val"]), dtype="float64").ravel()
    values = calculator.evaluate(MetricInputs(y_true=VAL_LABELS, y_pred=None, y_proba=scores))
    row = {str(name): str(value) for name, value in params.items()}
    row[CONFIG.metrics.primary] = values.get(CONFIG.metrics.primary, float("nan"))
    for name in RANKING_METRICS[1:]:
        row[name] = values.get(name, float("nan"))
    row["secondes"] = round(result.duration_seconds, 2)
    grid_rows.append(row)

grid_results = pd.DataFrame(grid_rows).sort_values(
    CONFIG.metrics.primary, ascending=ascending, na_position="last"
)
display(grid_results.round(4))

**Ce qu'il faut retenir**

- Le tri respecte le **sens** de la métrique (`direction: maximize` pour une PR AUC) : un tri ascendant par défaut classerait les pires détecteurs en premier.
- Une grille se lit aussi par sa **dispersion** : si toutes les combinaisons se tiennent en 0,01 de PR AUC, le détecteur est robuste et le réglage fin n'est pas le levier principal — les features le sont.
- Gare au sur-ajustement sur le split de validation : avec quelques dizaines de fraudes seulement, l'écart-type d'une PR AUC est de l'ordre de 0,01 à 0,03. Choisir le meilleur point d'une grille sur un seul split est un biais classique.
- Cas concret dans cette grille : `max_samples=0.5` s'affiche en tête, avec ~0,001 de PR AUC d'avance sur `max_samples=0.8` **à une seule graine**. Rejoué sur 5 graines, l'écart de moyenne reste de 0,001 alors que la dispersion entre graines est de 0,011 (0.5) et 0,003 (0.8) : le gagnant affiché est du bruit. Le réglage retenu est 0.8, qui divise la dispersion par 3,5 — donc qui rend la file d'alertes reproductible d'un ré-entraînement à l'autre.
- Règle pratique : ne retenir un point de grille que si son avance dépasse 2x la dispersion entre graines (section 4). En dessous, on tranche sur un critère non statistique — ici la stabilité, ailleurs le coût d'inférence ou la simplicité de maintenance.

## 6. Choix argumenté

| Critère | Lecture | Décision |
| --- | --- | --- |
| PR AUC (validation) | capacité de classement en forte imbalance | doit dépasser nettement la prévalence |
| Rappel au budget | fraude capturée à capacité constante | critère métier principal |
| Précision au budget | coût analyste par alerte | borne le volume d'alertes acceptable |
| Recouvrement du top-budget entre graines | reproductibilité de la file d'alertes | ≥ 0,85 avant mise en production |
| Coût d'entraînement / d'inférence | fenêtre de batch et latence temps réel | < 50 ms par transaction en scoring |

**Règle de décision retenue** : choisir le détecteur qui maximise le rappel **au budget déclaré**,
à précision au budget acceptable, puis vérifier la stabilité entre graines. Un gain de PR AUC
inférieur à la dispersion entre graines ne justifie pas un changement d'algorithme : il justifie un
travail sur les features (vélocité à fenêtre courte, indicateurs de manquants).